In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention2d(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.query = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.key   = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.value = nn.Conv2d(in_channels, in_channels, kernel_size=1)  # 保持通道数
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.size()
        N = H * W
        q = self.query(x).view(B, -1, N).permute(0, 2, 1)  # B x N x (C//8)
        k = self.key(x).view(B, -1, N)                    # B x (C//8) x N
        v = self.value(x).view(B, C, N)                  # B x C x N

        attn = torch.bmm(q, k)                           # B x N x N
        attn = attn / (q.shape[-1] ** 0.5)               # 缩放
        attn = torch.softmax(attn, dim=-1)

        out = torch.bmm(v, attn.permute(0, 2, 1))        # B x C x N
        out = out.view(B, C, H, W)
        return self.gamma * out + x

class Cecilia(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            SelfAttention2d(64)        # 注意大小写
        )
        # 注意：经过两次 MaxPool2d(2)，原始 224x224 -> 56x56
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x